In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 加载
logs = pd.read_csv('logs_clean.csv', parse_dates=['date', 'play_time'])
print("数据加载完成:", logs.shape)

数据加载完成: (18945380, 5)


In [2]:
end_date = pd.Timestamp('2009-05-05')

# 每个用户最后活跃日期
last_active = logs.groupby('user_id')['date'].max().reset_index()
last_active['last_active'] = pd.to_datetime(last_active['date'])
last_active['days_since_last'] = (end_date - last_active['last_active']).dt.days

# 流失定义：距离截止日超过30天未活跃
last_active['churn'] = (last_active['days_since_last'] > 30).astype(int)
print("流失分布:\n", last_active['churn'].value_counts())

流失分布:
 churn
0    831
1    160
Name: count, dtype: int64


In [3]:
obs_start = end_date - pd.Timedelta(days=60)
obs_logs = logs[(logs['date'] >= obs_start) & (logs['date'] <= end_date)]

# 基础特征
features = obs_logs.groupby('user_id').agg(
    active_days=('date', 'nunique'),
    total_plays=('date', 'count'),
    avg_daily_plays=('date', lambda x: len(x) / x.nunique() if x.nunique() > 0 else 0),
    avg_interval=('date', lambda x: x.sort_values().diff().dt.days.mean() if len(x) > 1 else np.nan),
    std_interval=('date', lambda x: x.sort_values().diff().dt.days.std() if len(x) > 1 else np.nan)
).reset_index()

features['avg_interval'] = features['avg_interval'].fillna(features['avg_interval'].median())
features['std_interval'] = features['std_interval'].fillna(0)

# 趋势特征：前30天 vs 后30天播放量比
mid_point = obs_start + pd.Timedelta(days=30)
first_half = obs_logs[obs_logs['date'] < mid_point].groupby('user_id').size().reset_index(name='plays_first')
second_half = obs_logs[obs_logs['date'] >= mid_point].groupby('user_id').size().reset_index(name='plays_second')
trend = first_half.merge(second_half, on='user_id', how='outer').fillna(0)
trend['trend_ratio'] = (trend['plays_second'] + 1) / (trend['plays_first'] + 1)

features = features.merge(trend[['user_id', 'trend_ratio']], on='user_id', how='left')
features['trend_ratio'] = features['trend_ratio'].fillna(1)

print("特征表样例:", features.head())

特征表样例:        user_id  active_days  total_plays  avg_daily_plays  avg_interval  \
0  user_000001           59         2221        37.644068      0.026577   
1  user_000002           46         1094        23.782609      0.047575   
2  user_000003           26          426        16.384615      0.124706   
3  user_000004           17          422        24.823529      0.128266   
4  user_000005           22          106         4.818182      0.561905   

   std_interval  trend_ratio  
0      0.163656     1.489362  
1      0.252326     0.683564  
2      0.816098     4.350000  
3      0.937415     0.081633  
4      1.834099     3.500000  


In [4]:
# 统一 user_id 为字符串
features['user_id'] = features['user_id'].astype(str)
last_active['user_id'] = last_active['user_id'].astype(str)

data = features.merge(last_active[['user_id', 'churn']], on='user_id', how='inner')
print(f"最终样本数: {len(data)}")
print("流失率:", data['churn'].mean())

最终样本数: 861
流失率: 0.03484320557491289


In [5]:
X = data.drop(['user_id', 'churn'], axis=1)
y = data['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

# 特征系数
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)
print(coef_df)

# 导出特征重要性
coef_df.to_csv('feature_importance.csv', index=False)
print("feature_importance.csv 导出完成")

              precision    recall  f1-score   support

           0       0.99      0.96      0.97       167
           1       0.36      0.67      0.47         6

    accuracy                           0.95       173
   macro avg       0.68      0.81      0.72       173
weighted avg       0.97      0.95      0.96       173

           feature  coefficient
2  avg_daily_plays     0.039250
1      total_plays    -0.006481
0      active_days    -0.320686
3     avg_interval    -0.464008
4     std_interval    -1.486081
5      trend_ratio    -5.151256
feature_importance.csv 导出完成



流失预警信号总结（按重要性）

- trend_ratio (系数 -5.15)：播放量衰减是最强流失信号，后30天腰斩的用户流失风险极高。
- std_interval (系数 -1.49)：流失用户的行为间隔变得更加均匀稀疏。
- avg_interval (系数 -0.46)：流失前可能有一段密集听歌期，然后快速断崖。
- active_days (系数 -0.32)：活跃天数越少，流失风险越高。


差异化召回策略设计

1. 趋势崩塌型（trend_ratio < 0.5）：
   推送历史最爱歌手新歌 / 专辑，退出弹窗“你常听的XXX出新作品了”。

2. 逐渐疏远型（active_days少，avg_interval在4-7天）：
   在用户活跃时段（参考24小时热力图）发送APP提醒，“午间放松歌单已更新”。

3. 暴饮暴食型（avg_daily_plays极高 + trend_ratio下降）：
   引导“少食多餐”，提供每日更新歌单（如“我的90年代最爱”），培养日常习惯。
